# Composition test

### Overview

**Input:** the per-review event sequences from 3.1
(`checkpoints/event_sequences.jsonl`) and the nested-label map
(`data/label_to_nested_mapping_reference.csv`).

**Pipeline:**
1. Setup and load every granularity.
2. Build a per-review event-proportion matrix and test each event type's between-review variance against a multinomial null (each review re-drawn i.i.d. from the corpus distribution at its own length).
3. Save one summary row per granularity.

**Read the effect size.** At ~6,500 reviews a negligible excess still produces a huge z; the pre-registered floor is on the overdispersion ratio.

## 1. Setup

In [3]:
# Cell 1: Imports

import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# Cell 2: Parameters

CHECKPOINTS = Path("checkpoints")
OUT_DIR     = Path("outputs_event_chains/composition")
OUT_DIR.mkdir(exist_ok=True)
SEQUENCES = CHECKPOINTS / "event_sequences.jsonl"

NESTED_PATH = Path("data/label_to_nested_mapping_reference.csv")
nested_df   = pd.read_csv(NESTED_PATH)
LABEL_TO_MR     = dict(zip(nested_df["Label"], nested_df["MR_nested"]))
LABEL_TO_AGENCY = dict(zip(nested_df["Label"], nested_df["Agency_nested"]))

# (name, sequence file, remap dict or None). Add or remove a line to change which
# granularities every test below runs over -- nothing else needs to change.
GRANULARITIES = [
    ("clusters_30",      SEQUENCES, None),
    ("MR_4",     SEQUENCES, LABEL_TO_MR),
    ("Agency_6", SEQUENCES, LABEL_TO_AGENCY),
]

RANDOM_STATE = 42

SUMMARY_PATH = OUT_DIR / "composition_signal_summary.csv"

**Pre-registered floor**
Set before running, and stated as an effect size rather than a significance test:
at ~6,500 reviews the z is uninformative, so the bar is a between-review spread large enough to matter, not one large enough to be "significant".

The floor is fixed on the between-review SD of a type's share, in percentage points, once sampling noise is subtracted. We treat a real spread below 5 pp on a review's most common event type as substantively negligible.
For a type with corpus share p and reviews of 17 tokens, the multinomial null variance is p(1-p)*mean(1/L), so an overdispersion of r implies a true SD of sqrt((r-1)*null)*100pp. On the dominant event type of every alphabet (share 0.30-0.69) that ~5 pp threshold lands at r = 1.20 (5.3 / 5.3 / 5.7 pp for clusters_30 / MR_4 / Agency_6). 

In [6]:
OVERDISPERSION_FLOOR = 1.20
N_PERMUTATIONS = 200

In [7]:
# Cell 3: Load alphabets

def load_chains(path, remap):
    with open(path) as f:
        seqs = [json.loads(line) for line in f]

    chains = []
    for s in seqs:
        tokens = [e["event"] for e in s["events"]]
        if remap is not None:
            # v1's 30 types are fully covered by the nested map; stop if that changes.
            missing = {t for t in tokens if t not in remap}
            assert not missing, f"tokens with no nested label: {missing}"
            tokens = [remap[t] for t in tokens]
        chains.append(tokens)
    return seqs, chains


loaded = {}
for name, path, remap in GRANULARITIES:
    seqs, chains = load_chains(path, remap)
    loaded[name] = {"seqs": seqs, "chains": chains}

    lengths = np.array([len(c) for c in chains])
    n_types = len({t for c in chains for t in c})
    print(f"{name:14s} {len(chains):,} reviews   {int(lengths.sum()):,} tokens   "
          f"{n_types:2d} types   median chain {int(np.median(lengths))}")

clusters_30    6,501 reviews   119,377 tokens   30 types   median chain 17
MR_4           6,501 reviews   119,377 tokens    4 types   median chain 17
Agency_6       6,501 reviews   119,377 tokens    6 types   median chain 17


## 2. multinomial null

For each granularity the code builds a per-review count matrix, turn it into proportions, and compare each event type's observed variance across reviews to the variance it shows when every review is re-drawn from one shared corpus distribution at that review's own chain length. 

Ratio is the **overdispersion**: 1.0 means the between-review spread is exactly what independent sampling from a single distribution produces.

In [8]:
# Cell 4: Overdispersion against the multinomial null

def composition_signal(chains, n_perm, seed):
    vocab = sorted({e for c in chains for e in c})
    idx   = {e: i for i, e in enumerate(vocab)}
    K     = len(vocab)

    counts = np.zeros((len(chains), K), dtype=np.int64)
    for i, c in enumerate(chains):
        for e in c:
            counts[i, idx[e]] += 1

    lengths = counts.sum(axis=1)
    keep    = lengths > 0   # every review with at least one event
    counts, lengths = counts[keep], lengths[keep]

    props    = counts / lengths[:, None]
    corpus_p = counts.sum(axis=0) / counts.sum()
    obs_var  = props.var(axis=0, ddof=1)

    # Null: re-draw each review from the corpus distribution at its own length.
    rng      = np.random.default_rng(seed)
    null_var = np.empty((n_perm, K))
    null_tot = np.empty(n_perm)
    for i in range(n_perm):
        sim = np.vstack([rng.multinomial(int(L), corpus_p) for L in lengths])
        sp  = sim / lengths[:, None]
        null_var[i] = sp.var(axis=0, ddof=1)
        null_tot[i] = null_var[i].sum()

    null_mean  = null_var.mean(axis=0)
    overdisp   = obs_var / null_mean
    true_sd_pp = np.sqrt(np.maximum(obs_var - null_mean, 0)) * 100

    per_type = pd.DataFrame({
        "type": vocab, "corpus_share": corpus_p,
        "overdispersion": overdisp, "true_sd_pp": true_sd_pp,
    }).sort_values("corpus_share", ascending=False)

    return {
        "n_reviews":        int(keep.sum()),
        "n_types":          K,
        "overdisp_median":  float(np.median(overdisp)),
        "overdisp_total":   float(obs_var.sum() / null_tot.mean()),
        "max_true_sd_pp":   float(true_sd_pp.max()),
        "per_type":         per_type,
    }


comp = {}
for name, _, _ in GRANULARITIES:
    r = composition_signal(loaded[name]["chains"], N_PERMUTATIONS, RANDOM_STATE)
    comp[name] = r
    verdict = "PASS" if r["overdisp_median"] > OVERDISPERSION_FLOOR else "fail"
    print(f"{name:14s} n_types {r['n_types']:2d}   overdispersion median "
          f"{r['overdisp_median']:.3f}   total {r['overdisp_total']:.3f}   "
          f"max true SD {r['max_true_sd_pp']:.1f} pp   [{verdict}]")

clusters_30    n_types 30   overdispersion median 1.043   total 1.029   max true SD 3.1 pp   [fail]
MR_4           n_types  4   overdispersion median 1.009   total 1.000   max true SD 2.1 pp   [fail]
Agency_6       n_types  6   overdispersion median 1.046   total 1.022   max true SD 2.0 pp   [fail]


## 3. Summary

In [9]:
# Cell 6: One row per granularity

rows = []
for name, _, _ in GRANULARITIES:
    r = comp[name]
    rows.append({
        "granularity":         name,
        "n_types":             r["n_types"],
        "n_reviews":           r["n_reviews"],
        "overdisp_median":     round(r["overdisp_median"], 3),
        "overdisp_total":      round(r["overdisp_total"], 3),
        "max_true_sd_pp":      round(r["max_true_sd_pp"], 2),
        "composition_verdict": "PASS" if r["overdisp_median"] > OVERDISPERSION_FLOOR else "fail",
    })

summary = pd.DataFrame(rows)
summary.to_csv(SUMMARY_PATH, index=False)

print(f"Saved {SUMMARY_PATH}")
print(f"PRE-REGISTERED FLOOR: overdisp_median > {OVERDISPERSION_FLOOR}")
print()
print(summary.to_string(index=False))

Saved outputs_event_chains/composition/composition_signal_summary.csv
PRE-REGISTERED FLOOR: overdisp_median > 1.2

granularity  n_types  n_reviews  overdisp_median  overdisp_total  max_true_sd_pp composition_verdict
clusters_30       30       6501            1.043           1.029            3.14                fail
       MR_4        4       6501            1.009           1.000            2.05                fail
   Agency_6        6       6501            1.046           1.022            1.98                fail
